<a href="https://colab.research.google.com/github/Oktaviani30/DeepLearningLanjut/blob/main/Nike_Oktaviani_Text_Generation_Menggunakan_Transformer_pert_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow==2.19.0
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:",
tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import requests
import numpy as np

# 1. Unduh teks Shakespeare
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
print(f"Panjang teks: {len(text)} karakter")

# 2. Bangun vocabulary (semua karakter unik)
vocab = sorted(set(text))
vocab_size = len(vocab)
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = np.array(vocab)

# 3. Encode teks menjadi integer
text_as_int = np.array([char_to_idx[c] for c in text])

# 4. Parameter Dataset
seq_length = 100
batch_size = 64
buffer_size = 10000

# 5. Buat dataset pasangan (input, target)
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)
dataset = dataset.shuffle(buffer_size).batch(batch_size, drop_remainder=True)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

Panjang teks: 1115394 karakter


In [3]:
# A. Positional Encoding
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model, max_len=5000):
        super().__init__()
        self.d_model = d_model
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pos_encoding = tf.Variable(tf.random.normal((1, max_len, d_model)))

    def call(self, x):
        seq_len = tf.shape(x)[1]
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = x + self.pos_encoding[:, :seq_len, :]
        return x

# B. Causal Multi-Head Attention
class CausalSelfAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.dense = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q, mask=None):
        batch_size = tf.shape(q)[0]
        q = self.wq(q); k = self.wk(k); v = self.wv(v)
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        # Causal Mask (Mencegah model melihat karakter masa depan)
        seq_len = tf.shape(q)[2]
        mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        scaled_attention_logits += (mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)
        output = tf.transpose(output, perm=[0, 2, 1, 3])
        output = tf.reshape(output, (batch_size, -1, self.d_model))
        return self.dense(output)

# C. Decoder Layer & GPT Model
def point_wise_feed_forward_network(d_model, dff):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(dff, activation='relu'),
        tf.keras.layers.Dense(d_model)
    ])

class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super().__init__()
        self.mha = CausalSelfAttention(d_model, num_heads)
        self.ffn = point_wise_feed_forward_network(d_model, dff)
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, x, training):
        attn_output = self.mha(x, x, x)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class GPT(tf.keras.Model):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, dff, max_len=1000, rate=0.1):
        super().__init__()
        self.pos_embedding = PositionalEmbedding(vocab_size, d_model, max_len)
        self.dec_layers = [DecoderLayer(d_model, num_heads, dff, rate) for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(rate)
        self.final_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, x, training):
        x = self.pos_embedding(x)
        x = self.dropout(x, training=training)
        for i in range(len(self.dec_layers)):
            x = self.dec_layers[i](x, training=training)
        return self.final_layer(x)

In [4]:
# Hyperparameter
model = GPT(vocab_size=vocab_size, d_model=128, num_layers=4, num_heads=8, dff=512, max_len=256)
optimizer = tf.keras.optimizers.Adam(0.001)
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

@tf.function
def train_step(inp, tar):
    with tf.GradientTape() as tape:
        predictions = model(inp, training=True)
        loss = loss_object(tar, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

# Proses Pelatihan (20 Epochs)
EPOCHS = 20
for epoch in range(EPOCHS):
    total_loss = 0
    for (batch, (inp, tar)) in enumerate(dataset):
        batch_loss = train_step(inp, tar)
        total_loss += batch_loss
    print(f'Epoch {epoch+1} Loss {total_loss/batch:.4f}')

Epoch 1 Loss 2.9078
Epoch 2 Loss 2.3808
Epoch 3 Loss 2.1650
Epoch 4 Loss 2.0091
Epoch 5 Loss 1.8961
Epoch 6 Loss 1.8146
Epoch 7 Loss 1.7520
Epoch 8 Loss 1.7040
Epoch 9 Loss 1.6676
Epoch 10 Loss 1.6351
Epoch 11 Loss 1.6078
Epoch 12 Loss 1.5862
Epoch 13 Loss 1.5675
Epoch 14 Loss 1.5504
Epoch 15 Loss 1.5356
Epoch 16 Loss 1.5211
Epoch 17 Loss 1.5083
Epoch 18 Loss 1.4992
Epoch 19 Loss 1.4879
Epoch 20 Loss 1.4789


In [5]:
def generate_text(model, start_string, length=100, temperature=1.0):
    input_ids = [char_to_idx.get(s, 0) for s in start_string]
    input_ids = tf.expand_dims(input_ids, 0)
    text_generated = []

    for i in range(length):
        predictions = model(input_ids, training=False)
        last_pred = predictions[:, -1, :] / temperature
        predicted_id = tf.random.categorical(last_pred, num_samples=1)
        predicted_id = tf.squeeze(predicted_id, axis=-1).numpy()[0]

        input_ids = tf.concat([input_ids, [[predicted_id]]], axis=1)
        text_generated.append(idx_to_char[predicted_id])

    return start_string + "".join(text_generated)

# Uji Coba Generate Teks
print(generate_text(model, start_string="ROMEO: ", length=200, temperature=0.8))

ROMEO: yet were was you.

VOLUMNIA:
No, fair,
Dispition to make the words.

PAULINA:
Upon, follow my git
I lore p wall an ie thowouged d t, mathere ore hougeratlly aly y t fore way, me fe t.
Ty walouly an at
